# 3D arm — final version (fixed axial augmentation, 5 seeds, saved predictions)

Re-runs the 3D arm after the augmentation fix in `src/common/data3d.py`
(rotation was applied in the y–z plane instead of the axial plane). Every run
now commits its test predictions, so the agreement-stratified analysis can be
computed for 3D as well.

Set **`PART`** in the next cell and run all. Each part fits inside Kaggle's
12 h cap:

| PART | runs | approx. time on a T4 |
|---|---|---|
| 1 | MoCo v2 pretraining (100 epochs), then Model B seeds 0–4 | ~8 h |
| 2 | Models A and C, binary — seeds 0–4 | ~7 h |
| 3 | Confident-label cohort, Model C and binary recipes — seeds 0–4 | ~2–3 h |

Parts can run in any order or in parallel sessions. Results are pushed to GitHub after **every seed**, and finished runs
are skipped, so if a session times out just run the same part again.

**Attach:** `ct-nodule-volumes`, `ct-nodule-metadata`. GPU on, Internet on,
Kaggle Secret `GITHUB_TOKEN` (repo scope).

In [ ]:
PART = 1   # 1, 2 or 3 -- see the table above
SEEDS = [0, 1, 2, 3, 4]

## 1. GPU check

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Locate the attached dataset

In [ ]:
import os, glob
for root in sorted(glob.glob("/kaggle/input/*")):
    print(root)
    for f in sorted(os.listdir(root))[:10]:
        p = os.path.join(root, f)
        sz = os.path.getsize(p)/1e9 if os.path.isfile(p) else None
        print("   ", f, f"({sz:.2f} GB)" if sz else "(folder)")

Paths are **discovered**, not hard-coded: Kaggle has more than one input layout
(`/kaggle/input/<slug>` on older sessions, `/kaggle/input/datasets/<user>/<slug>`
on newer ones), so the cell below searches for the files themselves.

The 3-class CSVs come from the **metadata** dataset: two files were truncated to
0 bytes when the volumes dataset was staged, so that copy is not trustworthy.
`qc_exclusions.csv` is used as the marker for the good copy because it is the
file that was empty there. Every CSV is asserted non-empty before training.


In [ ]:
import os, glob

def find_one(name, where="/kaggle/input"):
    """Locate a file anywhere under the input mounts, ignoring 0-byte copies."""
    hits = [p for p in glob.glob(f"{where}/**/{name}", recursive=True)
            if os.path.isfile(p) and os.path.getsize(p) > 0]
    if not hits:
        raise FileNotFoundError(
            f"no non-empty {name} under {where}; attached inputs: "
            f"{sorted(os.listdir(where))}")
    return sorted(hits, key=len)[0]

PACK = find_one("volumes_fp16.npy")
META = os.path.dirname(find_one("qc_exclusions.csv"))      # the clean 3-class CSVs
BIN  = os.path.dirname(find_one("binary_train.csv"))
print("volume pack :", PACK)
print("metadata dir:", META)
print("binary dir  :", BIN)

# config.py wants CT_DATA_ROOT/metadata/*.csv and /kaggle/input is read-only,
# so build a writable root of symlinks.
ROOT3C = "/kaggle/working/data_3class"
os.makedirs(f"{ROOT3C}/metadata", exist_ok=True)
for f in sorted(os.listdir(META)):
    if not f.endswith(".csv"):
        continue
    dst = f"{ROOT3C}/metadata/{f}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.unlink(dst)                       # drop stale links from earlier attempts
    os.symlink(f"{META}/{f}", dst)

os.environ["CT_DATA_ROOT"]   = ROOT3C
os.environ["CT_BINARY_ROOT"] = BIN
os.environ["CT_VOLUME_PACK"] = PACK

# fail loudly rather than training on a silently empty split
for d in (f"{ROOT3C}/metadata", BIN):
    print(d)
    for f in sorted(os.listdir(d)):
        if not f.endswith(".csv"):
            continue
        n = os.path.getsize(os.path.realpath(os.path.join(d, f)))
        assert n > 0, f"EMPTY CSV: {d}/{f}"
        print(f"   ok {f:32s} {n/1e6:7.2f} MB")


## 3. Clone the code

Private repo, so this needs a GitHub token with `repo` scope. Store it as a
Kaggle Secret (*Add-ons → Secrets*, name it `GITHUB_TOKEN`) so it is never
written into the notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
USER, REPO = "naazzarov", "CT-2D-2.5D-3D-DeepLearning"
!rm -rf /kaggle/working/repo; git clone -q https://{TOKEN}@github.com/{USER}/{REPO}.git /kaggle/working/repo
%cd /kaggle/working/repo
!ls src/3d

## 4. Verify the data

Asserts zero patient/nodule leakage, the agreed binary counts, and that volumes
load at the expected shape. Raises rather than warns.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/repo")
from src.common.data import load_splits, load_ssl_pool, assert_no_leakage, verify_binary_counts
from src.common.data3d import VolumeDataset, volume_train_statistics

s3 = load_splits("3class"); sb = load_splits("binary")
assert_no_leakage(s3, load_ssl_pool()); verify_binary_counts(sb)
assert_no_leakage(sb, load_ssl_pool(), reference=s3)
print("3-class:", {k: len(v) for k, v in s3.items()})
print("binary :", {k: len(v) for k, v in sb.items()})
mean, std = volume_train_statistics(s3["train"], limit=100)
print("volume intensity mean %.4f std %.4f" % (mean, std))
x, y = VolumeDataset(s3["val"], mean, std)[0]
print("sample shape", tuple(x.shape), "label", y)

## 5. Speed check

Expect roughly 0.3–0.8 min/epoch. An Apple M2 managed 13.5.

In [ ]:
import time, torch, sys
sys.path.insert(0, "/kaggle/working/repo/src/3d")
from model import ResNet10Classifier, n_params
dev="cuda"; m=ResNet10Classifier().to(dev)
opt=torch.optim.AdamW(m.parameters(), lr=1e-3); crit=torch.nn.CrossEntropyLoss()
x=torch.randn(16,1,104,72,80,device=dev); y=torch.randint(0,3,(16,),device=dev)
for _ in range(3): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize(); t=time.perf_counter()
for _ in range(10): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize(); per=(time.perf_counter()-t)/10
print(f"{per:.3f} s/step -> {1876/16*per/60:.1f} min/epoch")
print("params %.2fM" % (n_params(m)/1e6))

## 6. Training helper — skips finished runs, pushes after every seed

In [ ]:
import os, subprocess, glob, shutil
REPO = "/kaggle/working/repo"
GIT = ["git", "-c", "user.email=191228853+naazzarov@users.noreply.github.com",
       "-c", "user.name=Bahodir"]

def sh(cmd, check=True):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd, flush=True)
    return subprocess.run(cmd, cwd=REPO, check=check, shell=isinstance(cmd, str))

def push(msg):
    sh(GIT + ["add", "-A", "results"], check=False)
    if subprocess.run(GIT + ["diff", "--cached", "--quiet"], cwd=REPO).returncode == 0:
        print("nothing to commit"); return
    sh(GIT + ["commit", "-q", "-m", msg])
    for attempt in range(4):
        if subprocess.run(GIT + ["pull", "--rebase", "-q"], cwd=REPO).returncode == 0 and \
           subprocess.run(GIT + ["push", "-q"], cwd=REPO).returncode == 0:
            print("PUSHED"); return
    print("WARNING: push failed; results are still in", REPO)

def done(tag, seed):
    d = f"{REPO}/results/3d/{tag}/seed{seed}"
    return os.path.exists(f"{d}/result.json") and os.path.exists(f"{d}/test_probabilities.npy")

def train(tag, *args):
    for s in SEEDS:
        if done(tag, s):
            print(f"skip {tag} seed {s} (already pushed)"); continue
        sh(f"python src/3d/train_supervised.py --tag {tag} --seed {s} "
           f"--epochs 80 --patience 20 --batch-size 16 --workers 2 --eval-test "
           f"{' '.join(args)} 2>&1 | grep -vE '^ep +[0-9]+ \\|'")
        push(f"3D final: {tag} seed {s}")
    # The stratified analysis needs the agreement metadata columns; if they are
    # not in the attached metadata, the predictions are still pushed and the
    # analysis can be run locally later.
    sh(f"python scripts/stratify_by_agreement.py --reps 3d --tags {tag}", check=False)
    push(f"3D final: agreement analysis for {tag}")

## 7. Part 1 — MoCo v2 pretraining (fixed augmentation) and Model B

The previous encoder was trained with the y–z rotation, so pretraining is
repeated. The encoder (~57 MB) is not pushed to GitHub; it is copied to
`/kaggle/working/ckpt/` (kept with the notebook version) and reused if this
part is re-run after a timeout.

In [ ]:
if PART == 1:
    CKPT = "/kaggle/working/ckpt/moco_encoder_axialfix.pt"
    found = [p for p in glob.glob("/kaggle/input/**/moco_encoder_axialfix.pt", recursive=True)]
    if found and not os.path.exists(CKPT):
        os.makedirs(os.path.dirname(CKPT), exist_ok=True); shutil.copy(found[0], CKPT)
    if not os.path.exists(CKPT):
        sh("python src/3d/moco_pretrain.py --epochs 100 --batch-size 32 --workers 2")
        os.makedirs(os.path.dirname(CKPT), exist_ok=True)
        shutil.copy(f"{REPO}/results/3d/moco/moco_encoder.pt", CKPT)
        push("3D final: MoCo pretraining log (axial augmentation fix)")
    train("model_b_noaug", "--task 3class --init moco", f"--moco-ckpt {CKPT}", "--aug none")

## 8. Part 2 — Models A and C, binary

In [ ]:
if PART == 2:
    train("model_a_noaug", "--task 3class --init random --aug none")
    train("model_c_mild",  "--task 3class --init random --aug mild")
    train("binary_mild",   "--task binary --init random --aug mild")

## 9. Part 3 — confident-label cohort

Trains and tests only on nodules with concordant labels (no midpoint median,
at least three readers, no disagreement flag). This is the direct test of the
paper's thesis: if spatial context helps once label noise is removed, 3D
should pull ahead of 2D here.

In [ ]:
if PART == 3:
    train("model_c_mild_confident",  "--task 3class --init random --aug mild --cohort confident")
    train("binary_mild_confident",   "--task binary --init random --aug mild --cohort confident")

## 10. Summary

In [ ]:
!cd /kaggle/working/repo && git log --oneline -5 && ls results/3d
!cd /kaggle/working/repo && cat results/AGREEMENT.md 2>/dev/null | head -40